# Sentiment Analysis Demo Notebook
BMCS2003 Artificial Intelligence assignment.

**How to open this:** Anaconda Navigator -> Launch *Jupyter Notebook* (or in Anaconda Prompt: `conda activate sentiment-ai` then `jupyter notebook`), browse to `ml/notebooks/` and open `demo.ipynb`.

Run cells top to bottom with **Shift + Enter**. You do NOT need the Anaconda Prompt after the environment is created.


## 1. Make the `src` folder importable

In [ ]:
import os, sys, time
PROJECT = os.path.abspath(os.path.join(os.getcwd(), ".."))   # the ml/ folder
SRC = os.path.join(PROJECT, "src")
if SRC not in sys.path:
    sys.path.insert(0, SRC)
print("Project folder:", PROJECT)

## 2. Load the data and build the shared 80/20 split
Use `sample=400` first for a fast test, then set it to `None` for the full dataset.

In [ ]:
from split import get_splits
from preprocess import clean_text

DATASET = "cornell"   # "cornell" | "imdb" | "crawled"
SAMPLE  = 400         # None = use everything

X_train, X_test, y_train, y_test = get_splits(DATASET, SAMPLE)
print(len(X_train), "training reviews /", len(X_test), "test reviews")

X_train_c = [clean_text(t) for t in X_train]
X_test_c  = [clean_text(t) for t in X_test]
print("Example cleaned review:\n", X_train_c[0][:300])

## 3. Train the three models
Each cell prints the algorithm name plus Accuracy / Precision / Recall / F1.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from evaluation import evaluate_model

print(">>> ALGORITHM RUNNING: Multinomial Naive Bayes + Bag-of-Words")
nb = Pipeline([("bow", CountVectorizer(ngram_range=(1,2), min_df=2, max_features=50000)),
               ("clf", MultinomialNB(alpha=1.0))])
t0 = time.time(); nb.fit(X_train_c, y_train); nb_time = time.time() - t0
evaluate_model("Naive Bayes (BoW)", y_test, nb.predict(X_test_c), nb_time)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

print(">>> ALGORITHM RUNNING: Logistic Regression + TF-IDF")
logreg = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=50000, sublinear_tf=True)),
                   ("clf", LogisticRegression(max_iter=1000, C=5.0))])
t0 = time.time(); logreg.fit(X_train_c, y_train); lr_time = time.time() - t0
evaluate_model("Logistic Regression (TF-IDF)", y_test, logreg.predict(X_test_c), lr_time)

print(">>> ALGORITHM RUNNING: Linear SVM + TF-IDF")
svm = Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1,2), min_df=2, max_features=50000, sublinear_tf=True)),
                ("clf", LinearSVC(C=1.0))])
t0 = time.time(); svm.fit(X_train_c, y_train); svm_time = time.time() - t0
evaluate_model("Linear SVM (TF-IDF)", y_test, svm.predict(X_test_c), svm_time)

### Optional: deep learning model (DistilBERT)
Slow on CPU - skip during a live demo, or run it once beforehand on Google Colab with a GPU.

In [ ]:
# %run ../src/model3_distilbert.py --sample 2000 --epochs 1 --max_len 128

## 4. Demo: type in your own / fake reviews
Edit the `MY_REVIEWS` list below with anything you like (including fake or tricky reviews).
The table shows **which algorithm produced each prediction** and its confidence.

In [ ]:
import pandas as pd

MODELS = {
    "Naive Bayes (BoW)":            nb,
    "Logistic Regression (TF-IDF)": logreg,
    "Linear SVM (TF-IDF)":          svm,
}
LABELS = {0: "NEGATIVE", 1: "POSITIVE"}

MY_REVIEWS = [
    "Absolutely brilliant acting and a beautiful, moving story.",
    "The plot was predictable and dull, I nearly fell asleep.",
    "Best movie ever!!! Amazing amazing amazing, 10/10, buy tickets now!!!",   # fake / spammy
    "It was not good at all, despite the great cast.",                          # negation test
]

rows = []
for text in MY_REVIEWS:
    cleaned = clean_text(text)
    for name, model in MODELS.items():
        pred = int(model.predict([cleaned])[0])
        if hasattr(model, "predict_proba"):
            conf = f"{model.predict_proba([cleaned])[0][pred]:.1%}"
        else:
            conf = f"margin {model.decision_function([cleaned])[0]:+.2f}"
        rows.append({"review": text[:60] + ("..." if len(text) > 60 else ""),
                     "algorithm": name, "prediction": LABELS[pred], "confidence": conf})

pd.DataFrame(rows)

## 5. Why did it decide that? (pattern the model recognised)
Shows the words in your review that pushed the Logistic Regression model towards positive or negative - useful to explain 'how it recognises the pattern' during your demo.

In [ ]:
import numpy as np

def explain(text, model=logreg, top=8):
    vec, clf = model.named_steps["tfidf"], model.named_steps["clf"]
    cleaned = clean_text(text)
    x = vec.transform([cleaned])
    names = np.array(vec.get_feature_names_out())
    contrib = x.toarray()[0] * clf.coef_[0]
    idx = np.argsort(contrib)
    print("REVIEW:", text)
    print("PREDICTION:", LABELS[int(model.predict([cleaned])[0])], "(Logistic Regression + TF-IDF)")
    print("\nPushed POSITIVE:", [(names[i], round(contrib[i], 3)) for i in idx[::-1][:top] if contrib[i] > 0])
    print("Pushed NEGATIVE:", [(names[i], round(contrib[i], 3)) for i in idx[:top] if contrib[i] < 0])

explain("Best movie ever!!! Amazing amazing amazing, 10/10, buy tickets now!!!")

## 6. Interactive input box (nice for the live demo)

In [ ]:
text = input("Paste a review: ")
cleaned = clean_text(text)
for name, model in MODELS.items():
    print(f"{name:32s} -> {LABELS[int(model.predict([cleaned])[0])]}")

## 7. Save the models and the comparison chart
Writes `ml/models/*.joblib` and `ml/results/model_comparison.png` for your report.

In [ ]:
import joblib
from evaluation import plot_comparison

os.makedirs(os.path.join(PROJECT, "models"), exist_ok=True)
joblib.dump(nb,     os.path.join(PROJECT, "models", "naive_bayes.joblib"))
joblib.dump(logreg, os.path.join(PROJECT, "models", "tfidf_logreg.joblib"))
joblib.dump(svm,    os.path.join(PROJECT, "models", "tfidf_svm.joblib"))

try:
    plot_comparison()
    print("Chart written to ml/results/model_comparison.png")
except Exception as e:
    print("Chart step skipped:", e)